# Sliding-Window Reward Rate

Compute and visualize per-session reward rate as a function of trial index (and trial-start time) using `compute_sliding_reward_rate_from_nwb`.

In [ ]:
# Environment setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path('/root/capsule/src/aind_dft_ephys_analysis')
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import matplotlib.pyplot as plt

from nwb_utils import NWBUtils
from behavior_utils import (
    compute_sliding_reward_rate,
    compute_sliding_reward_rate_from_nwb,
)
print('Loaded modules.')

In [ ]:
# Sessions to analyze
sessions = [
    'ecephys_844034_2026-05-05_12-26-26_sorted_2026-05-13_16-51-38',
    'ecephys_844034_2026-05-06_12-31-42_sorted_2026-05-10_00-09-35',
    'ecephys_844034_2026-05-07_12-26-08_sorted_2026-05-10_22-15-43',
    'ecephys_844036_2026-05-04_16-06-43_sorted_2026-05-09_21-00-45',
    'ecephys_844036_2026-05-05_16-08-08_sorted_2026-05-19_17-42-51',
    'ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13',
]

# Sliding-window parameters
window        = 20          # trials
step          = 1           # one value per trial
causal        = False       # False -> centered; True -> trailing/causal
denominator   = 'window'    # or 'responded' to exclude no-response trials from denom
min_periods   = None        # None -> require full window (no edge values)

In [ ]:
# Load NWBs and compute sliding reward rate per session
results = {}
for sess in sessions:
    try:
        nwb = NWBUtils.read_ophys_or_behavior_nwb(session_name=sess)
        if nwb is None:
            print(f'[skip] no NWB for {sess}')
            continue
        try:
            res = compute_sliding_reward_rate_from_nwb(
                nwb,
                window=window,
                step=step,
                min_periods=min_periods,
                causal=causal,
                denominator=denominator,
            )
        finally:
            try:
                nwb.io.close()
            except Exception:
                pass
        results[sess] = res
        finite = np.isfinite(res['reward_rate'])
        mean_rr = float(np.nanmean(res['reward_rate'])) if finite.any() else float('nan')
        print(f'{sess}: n_trials={res["trial_index"].size}, mean_rr={mean_rr:.3f}')
    except Exception as e:
        print(f'[error] {sess}: {e}')

In [ ]:
# Per-session plots: reward rate vs trial index and distribution
# Use bin edges aligned to the discrete grid k/window so every possible
# reward-rate value falls in its own bin (no spurious empty bars).
bin_width = 1.0 / window
bins = np.arange(-bin_width / 2, 1.0 + bin_width, bin_width)

for sess, res in results.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2),
                             gridspec_kw={'width_ratios': [3, 1]})

    ax = axes[0]
    ax.plot(res['trial_index'], res['reward_rate'], color='tab:blue', lw=1.2)
    mu = float(np.nanmean(res['reward_rate']))
    ax.axhline(mu, color='k', ls='--', lw=0.8, label=f'mean={mu:.2f}')
    ax.set_xlabel('Trial index')
    ax.set_ylabel(f'Reward rate (window={window})')
    ax.set_title(f'{sess}')
    ax.set_ylim(0, 1)
    ax.legend(loc='lower right', fontsize=8)

    ax = axes[1]
    vals = res['reward_rate'][np.isfinite(res['reward_rate'])]
    if vals.size:
        ax.hist(vals, bins=bins, orientation='horizontal',
                color='tab:blue', alpha=0.75, edgecolor='k')
        med = float(np.median(vals))
        ax.axhline(med, color='tab:red', ls='--', lw=1.0, label=f'median={med:.2f}')
        ax.axhline(mu,  color='k',       ls='--', lw=0.8, label=f'mean={mu:.2f}')
        ax.legend(loc='lower right', fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Count')
    ax.set_title('Distribution')

    plt.tight_layout()
    plt.show()


In [ ]:
# Overlay: all sessions on one axis (reward rate vs trial index)
fig, ax = plt.subplots(figsize=(9, 4))
for sess, res in results.items():
    ax.plot(res['trial_index'], res['reward_rate'], lw=1.0, alpha=0.8, label=sess.split('_sorted')[0])
ax.set_xlabel('Trial index')
ax.set_ylabel(f'Reward rate (window={window})')
ax.set_ylim(0, 1)
ax.legend(fontsize=7, loc='lower right')
ax.set_title('Sliding-window reward rate across sessions')
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# Distribution of sliding-window reward rates
#   Left panel : per-session histograms overlaid (transparent)
#   Right panel: pooled histogram across all sessions
# =============================================================================
bins = np.linspace(0, 1, 31)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

# Per-session distributions
ax = axes[0]
cmap = plt.get_cmap('tab10')
for k, (sess, res) in enumerate(results.items()):
    vals = res['reward_rate']
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue
    ax.hist(vals, bins=bins, density=True, alpha=0.35,
            color=cmap(k % 10), label=sess.split('_sorted')[0])
    ax.axvline(np.median(vals), color=cmap(k % 10), ls='--', lw=0.8)
ax.set_xlabel(f'Reward rate (window={window})')
ax.set_ylabel('Density')
ax.set_title('Per-session distributions (dashed = median)')
ax.set_xlim(0, 1)
ax.legend(fontsize=7, loc='upper left')

# Pooled distribution
ax = axes[1]
pooled = np.concatenate([
    res['reward_rate'][np.isfinite(res['reward_rate'])]
    for res in results.values()
]) if results else np.array([])
if pooled.size:
    ax.hist(pooled, bins=bins, density=True, color='tab:gray', edgecolor='k', alpha=0.85)
    med = float(np.median(pooled))
    mu  = float(np.mean(pooled))
    ax.axvline(med, color='tab:red',  ls='--', lw=1.0, label=f'median={med:.2f}')
    ax.axvline(mu,  color='tab:blue', ls='--', lw=1.0, label=f'mean={mu:.2f}')
    ax.legend(loc='upper left', fontsize=8)
ax.set_xlabel(f'Reward rate (window={window})')
ax.set_ylabel('Density')
ax.set_title(f'Pooled across {len(results)} session(s) (n={pooled.size} windows)')
ax.set_xlim(0, 1)

plt.tight_layout(); plt.show()

# Per-session summary table
import pandas as pd
summary = pd.DataFrame([
    {
        'session': sess,
        'n_windows': int(np.isfinite(res['reward_rate']).sum()),
        'mean':   float(np.nanmean(res['reward_rate'])),
        'median': float(np.nanmedian(res['reward_rate'])),
        'std':    float(np.nanstd(res['reward_rate'])),
        'min':    float(np.nanmin(res['reward_rate'])),
        'max':    float(np.nanmax(res['reward_rate'])),
    }
    for sess, res in results.items()
])
summary


In [ ]:
# Compare causal vs centered window for a single example session
example_sess = sessions[0]
nwb = NWBUtils.read_ophys_or_behavior_nwb(session_name=example_sess)
try:
    rr_centered = compute_sliding_reward_rate_from_nwb(nwb, window=window, causal=False, min_periods=1)
    rr_causal   = compute_sliding_reward_rate_from_nwb(nwb, window=window, causal=True,  min_periods=1)
finally:
    try: nwb.io.close()
    except Exception: pass

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(rr_centered['trial_index'], rr_centered['reward_rate'], lw=1.2, label='centered')
ax.plot(rr_causal['trial_index'],   rr_causal['reward_rate'],   lw=1.2, label='causal (trailing)')
ax.set_xlabel('Trial index'); ax.set_ylabel('Reward rate'); ax.set_ylim(0, 1)
ax.set_title(f'Centered vs causal window ({example_sess.split("_sorted")[0]})')
ax.legend(); plt.tight_layout(); plt.show()